# 06 · MoE Expert Routing

- **Objectives**: build a Mixture-of-Experts model, run a forward pass, extract per-layer expert utilization, and visualize routing patterns. Compare the two routers (softmax top-k vs DeepSeek-V3 sigmoid).
- **Requirements**: 1 GPU (CPU also works).
- **Runtime**: ~10 seconds.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Build an MoE model

Setting `num_experts > 0` turns the MLP block into a `MoEMLP` — a router that sends each token to `moe_top_k` of `num_experts` experts, then a weighted sum of expert outputs.

In [ ]:
from kempnerforge.config.schema import ModelConfig
from kempnerforge.model.transformer import Transformer


def build_moe(router: str) -> Transformer:
    config = ModelConfig(
        dim=128,
        n_layers=4,
        n_heads=4,
        n_kv_heads=4,
        vocab_size=256,
        max_seq_len=64,
        num_experts=8,
        moe_top_k=2,
        moe_frequency=1,  # every layer is MoE
        moe_router=router,
    )
    return Transformer(config).to(device)


model = build_moe(router="softmax_topk")
print("MoE model built — every block uses MoEMLP")
print(f"MLP type: {type(model.layers['0'].mlp).__name__}")

## Forward pass

Run a batch through the model. Each layer's router makes a discrete routing decision per token during the forward pass.

In [ ]:
torch.manual_seed(0)
batch, seq_len = 4, 32
n_tokens = batch * seq_len
tokens = torch.randint(0, 256, (batch, seq_len), device=device)

with torch.no_grad():
    _ = model(tokens)

print(f"Forward pass complete — {n_tokens} tokens through {model.config.n_layers} MoE layers")

## Inspect per-layer expert utilization

`Transformer.get_expert_counts()` returns `{layer_idx: counts}` where `counts[i]` is the number of tokens routed to expert `i` in that layer. Per token, `top_k` experts are selected, so the total across experts = `n_tokens × top_k`.

In [ ]:
expert_counts = model.get_expert_counts()

expected_total = n_tokens * model.config.moe_top_k
print(f"Expected total (n_tokens × top_k) = {expected_total}")
print()
for layer_idx, counts in expert_counts.items():
    counts_list = counts.tolist()
    ratio = min(counts_list) / max(counts_list) if max(counts_list) > 0 else 0
    print(
        f"  Layer {layer_idx}: sum={int(sum(counts_list))}, "
        f"per-expert={counts_list}, min/max={ratio:.2f}"
    )

## Visualize: one bar chart per layer

Perfectly uniform load balancing would give every expert the same count: `n_tokens × top_k / num_experts`. Deviations from this mean some experts are over-used (hot) and others are under-used (cold) — that's what the auxiliary balance loss is designed to penalize.

In [ ]:
uniform = expected_total / model.config.num_experts
n_layers = len(expert_counts)
fig, axes = plt.subplots(1, n_layers, figsize=(3 * n_layers, 3.5), sharey=True)
for ax, (layer_idx, counts) in zip(axes, expert_counts.items()):
    counts_cpu = counts.detach().cpu().numpy()
    ax.bar(range(model.config.num_experts), counts_cpu)
    ax.axhline(uniform, linestyle="--", color="red", alpha=0.6, label="uniform")
    ax.set_title(f"Layer {layer_idx}")
    ax.set_xlabel("Expert")
axes[0].set_ylabel("Tokens routed")
axes[0].legend(loc="upper right", fontsize=8)
plt.suptitle("MoE expert utilization (softmax_topk router, untrained)", y=1.02)
plt.tight_layout()
plt.show()

## Compare softmax vs sigmoid routers

KempnerForge supports two routers:
- **`softmax_topk`** (Mixtral-style): softmax over expert logits → top-k → renormalize. Auxiliary load-balancing loss via Switch-Transformer style penalty.
- **`sigmoid_topk`** (DeepSeek-V3 style): sigmoid on expert scores, bias-based load balancing — maintains an EMA of expert utilization, adjusts a learnable per-expert bias to keep load even without an explicit auxiliary loss.

Both are available as a 1-line config change.

In [ ]:
torch.manual_seed(0)
model_sigmoid = build_moe(router="sigmoid_topk")
with torch.no_grad():
    _ = model_sigmoid(tokens)
counts_sigmoid = model_sigmoid.get_expert_counts()


def spread(counts):
    arr = np.array(counts.detach().cpu().tolist())
    return arr.std() / arr.mean() if arr.mean() > 0 else 0


print(f"{'Layer':>6} {'softmax_topk CV':>18} {'sigmoid_topk CV':>18}")
print("-" * 44)
for layer_idx in sorted(expert_counts):
    cv_softmax = spread(expert_counts[layer_idx])
    cv_sigmoid = spread(counts_sigmoid[layer_idx])
    print(f"{layer_idx:>6} {cv_softmax:>18.3f} {cv_sigmoid:>18.3f}")

Lower coefficient of variation (CV) = more balanced routing. At random init both routers are near-uniform by chance. The difference appears after training, where the sigmoid router's bias mechanism keeps experts balanced without needing to tune an auxiliary loss weight.

## Notes

- **Capacity factor**: with `moe_capacity_factor > 0`, overflow tokens have their routing weights zeroed (token dropping). Counts still include them, but they contribute nothing to the output.
- **Shared experts**: DeepSeek-V3 style can add `moe_shared_experts > 0` — dense experts that process every token (they don't appear in `expert_counts`).
- **Auxiliary loss**: use `model.get_moe_aux_loss()` to collect the balance penalty and add it (with a weight) to your training loss.
- **Expert parallelism (EP)**: for serious MoE training, `distributed.ep > 1` shards experts across ranks with all-to-all dispatch. See `benchmarks/moe_expert_parallel/`.